In [1]:
# Load environment variables and verify the project setup.
import sys
from pathlib import Path

# Find the repo root (the folder containing env_checker.py) and make it importable.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "env_checker.py").exists())
sys.path.insert(0, str(ROOT))

# Load .env into the environment for this session.
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ModuleNotFoundError:
    print("python-dotenv not installed yet — run: uv add python-dotenv")

# Verify .env variables and required packages.
from env_checker import run_checks
run_checks()

Environment variables (from .env.example)
  ✓ OPENAI_API_KEY  — set
  ✓ ANTHROPIC_API_KEY  — set
  ✓ LANGSMITH_TRACING  — set
  ✓ LANGSMITH_ENDPOINT  — set
  ✓ LANGSMITH_API_KEY  — set
  ✓ LANGSMITH_PROJECT  — set
  ✓ CHROMA_PERSIST_DIR  — set

Required packages (from pyproject.toml)
  ✓ beautifulsoup4  — installed (4.14.3)
  ✓ chromadb  — installed (1.5.9)
  ✓ langchain  — installed (1.3.2)
  ✓ langchain-chroma  — installed (1.1.0)
  ✓ langchain-classic  — installed (1.0.7)
  ✓ langchain-community  — installed (0.4.2)
  ✓ langchain-core  — installed (1.4.0)
  ✓ langchain-experimental  — installed (0.4.2)
  ✓ langchain-openai  — installed (1.2.2)
  ✓ lxml  — installed (6.1.1)
  ✓ onnxruntime  — installed (1.19.2)
  ✓ pypdf  — installed (6.12.2)
  ✓ python-dotenv  — installed (1.2.2)
  ✓ rank-bm25  — installed (0.2.2)
  ✓ rapidfuzz  — installed (3.14.5)
  ✓ ipykernel  — installed (7.2.0)
  ✓ jupyterlab  — installed (4.5.7)

✓ All checks passed.


True

# Context Precision

**Context precision** measures whether the *relevant* retrieved contexts are ranked **near the top**. A retriever that returns a relevant chunk at rank 1 scores higher than one that buries it at rank 5.

We judge each retrieved context as relevant/not (LLM-as-judge, given a reference answer), then average Precision@k over the ranks where a relevant context appears — the same idea as RAGAS's `LLMContextPrecisionWithReference`.

## Example: retrieved contexts in rank order

In [2]:
question = "Who owns the Low-Level Design (LLD) and what does it contain?"
reference = ("LLD is owned by senior developers / tech lead and contains the "
             "detailed internal design of each component.")

# As returned by a retriever, best-first:
retrieved_contexts = [
    "Low-Level Design (LLD): detailed internal design of each component; owner: Senior Developers / Tech Lead.",  # relevant
    "Release Notes summarize new features and bug fixes for a release.",                                          # not relevant
    "LLD specifies class/module structure, methods, data structures and database fields.",                        # relevant
]

## Judge relevance of each context

In [3]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

class Relevance(BaseModel):
    relevant: bool = Field(description="True if the context helps answer the question")
    reason: str

judge = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(Relevance)

PROMPT = """Decide whether the CONTEXT is relevant/useful for answering the QUESTION
(a REFERENCE answer is given for guidance).

QUESTION: {question}
REFERENCE: {reference}
CONTEXT: {context}
"""

verdicts = []
for i, ctx in enumerate(retrieved_contexts, 1):
    r = judge.invoke(PROMPT.format(question=question, reference=reference, context=ctx))
    verdicts.append(r.relevant)
    print(f"[{i}] relevant={r.relevant} :: {ctx[:60]}...")

[1] relevant=True :: Low-Level Design (LLD): detailed internal design of each com...


[2] relevant=False :: Release Notes summarize new features and bug fixes for a rel...


[3] relevant=True :: LLD specifies class/module structure, methods, data structur...


## Score

Average of Precision@k at each rank holding a relevant context — rewards ranking relevant contexts early.

In [4]:
def context_precision(verdicts):
    hits, precisions = 0, []
    for k, rel in enumerate(verdicts, start=1):
        if rel:
            hits += 1
            precisions.append(hits / k)   # Precision@k at this relevant item
    return sum(precisions) / len(precisions) if precisions else 0.0

print(f"verdicts (rank order): {verdicts}")
print(f"Context Precision = {context_precision(verdicts):.3f}")

verdicts (rank order): [True, False, True]
Context Precision = 0.833
